# FallSense sensor baseline — Colab handoff
Outer LOSO + inner subject-grouped calibration/threshold。真實資料與 checkpoint 只留在使用者 Drive。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
from pathlib import Path

MODE = "quick"  # quick | full
REPO = Path("/content/drive/MyDrive/FallSense/repo")
HANDOFF = Path("/content/drive/MyDrive/FallSense")
RUN_ID = f"gbm-{MODE}-colab"
print({"mode": MODE, "T4_budget_gb": 16, "repo": str(REPO), "handoff": str(HANDOFF)})
assert REPO.joinpath("pyproject.toml").exists(), (
    "先把 repo 放到 Drive FallSense/repo；目前尚無 git remote 可 clone"
)

In [ ]:
%cd {REPO}
!python -m pip install -q '.[train,plot]'
import subprocess

raw_dir = REPO / "tests/fixtures/upfall_synthetic" if MODE == "quick" else HANDOFF / "data/raw"
window_config = REPO / (
    "configs/windowing_quick_20_10.yaml" if MODE == "quick" else "configs/windowing_50_25.yaml"
)
processed = HANDOFF / f"processed/{RUN_ID}"
result_dir = HANDOFF / f"results/{RUN_ID}"
result_file = result_dir / "results.json"
print({"raw_exists": raw_dir.exists(), "estimated_peak_ram_gb": 2.5 if MODE == "full" else 0.5})

In [ ]:
if result_file.exists():
    print("RESUME: results.json 已存在，跳過重跑", result_file)
else:
    subprocess.run(
        [
            "python",
            "scripts/make_windows.py",
            "--raw-dir",
            str(raw_dir),
            "--config",
            str(window_config),
            "--output-dir",
            str(processed),
            "--split",
            "loso",
        ],
        check=True,
    )
    data_manifest = (
        REPO / "tests/fixtures/upfall_synthetic/fixture_manifest.json"
        if MODE == "quick"
        else HANDOFF / "data/manifest.json"
    )
    status = "QUICK-MODE-ONLY" if MODE == "quick" else "RUN"
    subprocess.run(
        [
            "python",
            "scripts/train_gbm.py",
            "--windows",
            str(processed / "windows.npz"),
            "--split-manifest",
            str(processed / "split_manifest.json"),
            "--data-manifest",
            str(data_manifest),
            "--output-dir",
            str(result_dir),
            "--run-id",
            RUN_ID,
            "--status",
            status,
        ],
        check=True,
    )
    print("完成並保存", result_file)